# Programming for Linguistics - Lesson 5: N-grams and Frequency Analysis

In this lesson, we will explore **N-grams**, a fundamental concept in Computational Linguistics and Natural Language Processing (NLP). 

An **N-gram** is a contiguous sequence of *n* items from a given sample of text or speech. 
*   **Unigrams**: Single words (`n=1`)
*   **Bigrams**: Pairs of consecutive words (`n=2`)
*   **Trigrams**: Triplets of consecutive words (`n=3`)

N-grams are used for various tasks, such as:
*   **Language Modeling**: Predicting the next word in a sentence (like autocomplete).
*   **Collocation Extraction**: Identifying common phrases or idioms.
*   **Author Attribution**: Analyzing the unique stylistic patterns of an author.

---


## Theory: Zipf's Law and Lexical Distribution
In almost any large corpus, a few words (like 'the', 'of', 'and') are extremely frequent, while the vast majority of words occur only once or twice (the 'Long Tail'). This is **Zipf's Law**. In NLP, we often remove 'Stopwords' (these high-frequency, low-info words) to focus on the semantically rich content of a text.


## Step 1: Environment Setup

To begin, we need to import some utility functions and ensure our Python environment can find them. We add the project directory to `sys.path` so that our custom modules (like `utils.py`) can be imported properly.


In [ ]:
from utils import infile_outsents_comp, get_bigrams_freqs

In [ ]:
import sys
sys.path.append('/Users/luca/Gits/pfl_2026_unicatt/')

In [ ]:
sys.path

## Step 2: Loading the Corpus

We load the text of Bram Stoker's *Dracula* (Project Gutenberg ID 345). 

The function `infile_outsents_comp` is a custom utility that:
1. Reads the raw text file.
2. Normalizes the content.
3. Splits it into a list of sentences, where each sentence is a **list of tokens** (words and punctuation).


In [ ]:
text = infile_outsents_comp('assets/pg345.txt')
# text

## Step 3: The "Zip" Trick for N-grams

Generating N-grams manually with loops can be complex. Python provides a very elegant way to do this using the `zip()` function and **list slicing**.

### How it works:
If we have a list `a = [1, 2, 3, 4]`, then `a[1:]` is `[2, 3, 4]`. 
When we `zip(a, a[1:])`, Python pairs them up:
*   `1` with `2` 
*   `2` with `3` 
*   `3` with `4` 

This creates our bigrams! For trigrams, we would zip `a`, `a[1:]`, and `a[2:]`.


In [ ]:
a = 'a b c d e f g'.split()
b = "1 2 3 4 5 6 7".split()
len(b) == len(a)

for j,k in zip(a, a[1:]):
    print(j,k)

In [ ]:
sent = text[34]

#Ngrams base 2 or bigrams
for w1,w2, w3 in zip(sent, sent[1:], sent[2:] ):
    print(w1, w2, w3)

## Step 4: Building a Frequency Analyzer

Now that we can generate bigrams, we want to see which ones are the most common in *Dracula*. However, many frequent bigrams are not very informative (e.g., "of the", "in a"). These often involve **stopwords** (highly frequent words with little semantic value) or **punctuation**.

### Logic of `get_bigrams_freqs`:
1.  **Filtering**: We only keep bigrams where *neither* word is punctuation or a stopword.
2.  **Counting**: We use a `defaultdict(int)`. This is a special dictionary that automatically initializes a new key with `0` if it doesn't exist, making counting much cleaner.
3.  **Sorting**: We sort the resulting dictionary by frequency in descending order.


In [ ]:
# typewriter
# machine gun

In [ ]:
from collections import defaultdict
from string import punctuation
punctuation += "”“’--"
from nltk.corpus import stopwords
en_stops = stopwords.words('english')

def get_bigrams_freqs(corpus, threshold= 5):
    # populate the dictionary with bigrams frequencies, where bigrams are keys and their value is the frequency
    # out = {}
    out = defaultdict(int)
    for text in corpus:
        for w1,w2 in zip(text, text[1:]):
            if w1 not in punctuation and w2 not in punctuation and w1 not in en_stops and w2 not in en_stops:
                bigram = f'{w1}_{w2}'
                # if bigram in out:
                #     out[bigram] += 1
                # else:
                #     out[bigram] = 1
                out[bigram] +=1
    sorted_freqs = sorted(out.items(), key=lambda x : x[1], reverse=True )
    return [t for t in sorted_freqs if t[1] >= threshold]


get_bigrams_freqs(text, threshold=2)



### Cleaning Data: Punctuation and Stopwords

To get meaningful results, we often need to expand the default list of punctuation provided by Python to include "smart" quotes and other symbols found in literary texts.


In [ ]:
from string import punctuation
punctuation += "”“’"
punctuation

## Step 5: Advanced Python - Lambda and Sorting

Dictionaries in Python are not sorted by default. To sort them by their **values** (the frequencies), we use the `sorted()` function.

We need to tell Python: "Don't sort by the word (the key), sort by the number (the value)". 

A **Lambda function** is a small, anonymous function defined in one line. For example, `lambda x: x[1]` tells Python to look at the second element of each pair (the frequency) when sorting.


In [ ]:
d = {"a" : 345, 
     "b" : 846,
     "c" : 23}
def sorting_ints(tup):
    return tup[1]

sorted(d.items(), key=lambda x : x[1], reverse=True )


In [ ]:
lambda_ints = lambda x : x[1]
lambda_ints(("a", 754))

In [ ]:
d = defaultdict(int)
# d = {}
d["of"] +=1
# d.items()

In [ ]:
vocab = set()
for sent in text:
    vocab.update(sent)
vocab = {word:i for i, word in enumerate(vocab)}
vocab

In [ ]:
sample = text[45][:15]
v = set(sample)
len(v)

In [ ]:
m = [[0,0,0,0,0,0,0,0,0,0,0,0,0],
     [0,0,0,0,0,0,0,0,0,0,0,0,0],
     [0,0,0,0,0,0,0,0,0,0,0,0,0], 
]

In [ ]:
m[0][2] +=1

In [ ]:
{word:i for i,word in enumerate(v)}   

In [ ]:
def co_occurrence_matrix(txt):
    vocab = set()
    for sent in txt:
        vocab.update(sent)
    vocab = {word:i for i, word in enumerate(vocab)}
    mtx = np.zeros((len(vocab), len(vocab)))
    for sent in txt:
        for w1,w2 in zip(sent, sent[1:]):
            iw1 = vocab[w1]
            iw2 = vocab[w2]
            mtx[iw1,iw2] +=1
    return mtx

m = co_occurrence_matrix(text)
sum(m[162])
    

In [ ]:
import numpy as np

np.array((1,2))

In [ ]:
zeros_array = np.zeros((3, 5))
zeros_array[1,2] = 10
zeros_array

In [ ]:
np.ones((3, 5))

## Exercises
Complete the following tasks to practice the concepts covered in this lesson. Aim for efficient, readable code.


### Exercise 1
Load the Dracula corpus and find the 20 most frequent words. How many of them are 'function words' (articles, prepositions) vs 'content words' (nouns, verbs)?


### Exercise 2
Create a list of 10 common English stopwords. Write a script to remove these from the Dracula token list and recalculate the top 20 words.


### Exercise 3
Calculate what percentage of the total text in Dracula is composed of the top 10 most frequent words.


### Exercise 4
Find all words in the corpus that are longer than 12 characters and occur at least 5 times. List them alphabetically.


### Exercise 5
Linguistic Analysis: Count how many times the word 'blood' appears in Dracula vs the word 'love'. Does the ratio surprise you?
